# Customer Churn Intelligence — Logistic Regression

## Objective

Train an **interpretable linear baseline** with and without class weighting. Evaluate on the **validation set only** — the final test set is not used.

**Stage:** Step 11 — Logistic Regression (no SMOTE, threshold tuning, calibration, SHAP, tree models, or test-set evaluation).

### Why Logistic Regression?
- **Simple interpretable baseline** — coefficients relate directly to feature influence (after preprocessing).
- **Produces probabilities** — supports retention prioritization and later threshold/calibration work.
- **Good reference before nonlinear models** — if a linear model performs reasonably, tree/boosted models show how much nonlinearity helps.

### Why not Random Forest / XGBoost yet?
We first need to see how well a **simple linear model** performs on this problem before adding nonlinear complexity.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42

## 1. Load Train / Validation Data

In [ ]:
split = load_split_from_manifest()

X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")
print(f"Validation churn prevalence: {y_val.mean():.2%}")

## 2. Train Two Logistic Regression Versions

In [ ]:
model_configs = {
    "LogisticRegression": {"class_weight": None},
    "LogisticRegression (balanced)": {"class_weight": "balanced"},
}

pipelines = {}
for name, model_kwargs in model_configs.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "model",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **model_kwargs),
            ),
        ]
    )
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print(f"Fitted: {name}")

## 3. Validation Metrics

In [ ]:
def evaluate_model(name: str, pipeline: Pipeline, X_val: pd.DataFrame, y_val: pd.Series) -> dict:
    y_pred = pipeline.predict(X_val)
    y_proba = pipeline.predict_proba(X_val)[:, 1]
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_val, y_pred), 4),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_val, y_pred, zero_division=0), 4),
        "ROC_AUC": round(roc_auc_score(y_val, y_proba), 4),
        "PR_AUC": round(average_precision_score(y_val, y_proba), 4),
        "Predicted_Churners": int(y_pred.sum()),
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


results = {name: evaluate_model(name, pipe, X_val, y_val) for name, pipe in pipelines.items()}

metrics_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in {"y_pred", "y_proba"}} for r in results.values()]
)
metrics_df

In [ ]:
for name, result in results.items():
    cm = confusion_matrix(y_val, result["y_pred"])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"],
    )
    print(f"\nConfusion Matrix — {name}")
    display(cm_df)

## 4. Predicted Probability Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (name, result) in zip(axes, results.items()):
    ax.hist(result["y_proba"], bins=30, color="#4C72B0", edgecolor="white")
    ax.axvline(0.5, color="#DD8452", linestyle="--", label="Default threshold 0.5")
    ax.set_title(f"Predicted P(Churn) — {name}")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Customer count")
    ax.legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "07_logistic_regression_probability_distributions.png", dpi=120)
plt.show()

In [ ]:
proba_summary = pd.DataFrame(
    {
        "Model": list(results.keys()),
        "Min P(Churn)": [r["y_proba"].min() for r in results.values()],
        "Mean P(Churn)": [r["y_proba"].mean() for r in results.values()],
        "Max P(Churn)": [r["y_proba"].max() for r in results.values()],
        "Predicted churners (threshold=0.5)": [r["Predicted_Churners"] for r in results.values()],
    }
).round(4)
proba_summary

## 5. Model Comparison Table (incl. DummyClassifier)

In [ ]:
lr_rows = [{k: v for k, v in r.items() if k not in {"y_pred", "y_proba"}} for r in results.values()]
lr_df = pd.DataFrame(lr_rows)

if COMPARISON_PATH.exists():
    existing = pd.read_csv(COMPARISON_PATH)
    existing = existing[~existing["Model"].isin(lr_df["Model"])]
    comparison_df = pd.concat([existing, lr_df], ignore_index=True)
else:
    comparison_df = lr_df

comparison_df = comparison_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
]
comparison_df.to_csv(COMPARISON_PATH, index=False)

print(f"Updated: {COMPARISON_PATH}")
comparison_df

## 6. Which Logistic Regression Version Is Better?

Both versions **substantially beat** the DummyClassifier on Recall, F1, ROC-AUC, and PR-AUC.

| Version | Strengths | Trade-offs |
|---------|-----------|------------|
| **Default** | Higher **Accuracy** and **Precision**; slightly higher ROC-AUC | Lower **Recall** — misses more churners |
| **Balanced** | Higher **Recall**, **F1**, and **PR-AUC**; flags more churners | Lower Precision and Accuracy; more false positives |

**Recommendation for this churn problem:** `LogisticRegression(class_weight='balanced')` is the stronger validation reference because retention campaigns prioritize **finding churners** (Recall) while maintaining competitive ranking metrics (PR-AUC, ROC-AUC). The default model is preferable only if false-positive contact cost is extremely high.

Threshold optimization (not done here) may further improve the precision/recall trade-off.

## Summary

- **Training:** 4,930 rows | **Validation:** 1,056 rows | **Test:** not used
- **Preprocessor:** same `ColumnTransformer` as Step 9, fitted inside each pipeline on training data
- **Both LR models** outperform DummyClassifier on all metrics except raw Accuracy (dummy ~73.5% by always predicting No)
- **Best LR version (validation):** `LogisticRegression(class_weight='balanced')` on Recall/F1/PR-AUC

**Next step (not performed here):** Random Forest comparison.